In [13]:
!pip install psycopg2-binary



[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [14]:
!pip install pandas


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [15]:
import os
import pandas as pd



In [16]:
import psycopg2

In [17]:
rawdir = './data/raw/'
processed = './data/processed/'

In [18]:
yanki_ecommerce = os.path.join(rawdir, 'yanki_ecommerce.csv')
yanki_ecommerce

'./data/raw/yanki_ecommerce.csv'

In [19]:
ykdf = pd.read_csv(yanki_ecommerce)

### Check for missing values

In [20]:
ykdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Order_ID            1000 non-null   object 
 1   Customer_ID         1000 non-null   object 
 2   Customer_Name       1000 non-null   object 
 3   Product_ID          1020 non-null   object 
 4   Product_Name        1020 non-null   object 
 5   Brand               1020 non-null   object 
 6   Category            1020 non-null   object 
 7   Price               1020 non-null   float64
 8   Quantity            1020 non-null   int64  
 9   Total_Price         1020 non-null   float64
 10  Order_Date          1020 non-null   object 
 11  Shipping_Address    1020 non-null   object 
 12  City                1020 non-null   object 
 13  State               1019 non-null   object 
 14  Country             1020 non-null   object 
 15  Postal_Code         1020 non-null   int64  
 16  Email 

In [21]:
ykdf.columns

Index(['Order_ID', 'Customer_ID', 'Customer_Name', 'Product_ID',
       'Product_Name', 'Brand', 'Category', 'Price', 'Quantity', 'Total_Price',
       'Order_Date', 'Shipping_Address', 'City', 'State', 'Country',
       'Postal_Code', 'Email', 'Phone_Number', 'Payment_Method',
       'Transaction_Status'],
      dtype='object')

In [131]:
ykdf.dropna(subset=['Order_ID', 'Customer_ID','State'],inplace=True)
ykdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 999 entries, 2 to 1019
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Order_ID            999 non-null    object        
 1   Customer_ID         999 non-null    object        
 2   Customer_Name       999 non-null    object        
 3   Product_ID          999 non-null    object        
 4   Product_Name        999 non-null    object        
 5   Brand               999 non-null    object        
 6   Category            999 non-null    object        
 7   Price               999 non-null    float64       
 8   Quantity            999 non-null    int64         
 9   Total_Price         999 non-null    float64       
 10  Order_Date          999 non-null    datetime64[ns]
 11  Shipping_Address    999 non-null    object        
 12  City                999 non-null    object        
 13  State               999 non-null    object        
 14

In [133]:
ykdf['Order_Date'] = pd.to_datetime(ykdf['Order_Date'])
ykdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 999 entries, 2 to 1019
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Order_ID            999 non-null    object        
 1   Customer_ID         999 non-null    object        
 2   Customer_Name       999 non-null    object        
 3   Product_ID          999 non-null    object        
 4   Product_Name        999 non-null    object        
 5   Brand               999 non-null    object        
 6   Category            999 non-null    object        
 7   Price               999 non-null    float64       
 8   Quantity            999 non-null    int64         
 9   Total_Price         999 non-null    float64       
 10  Order_Date          999 non-null    datetime64[ns]
 11  Shipping_Address    999 non-null    object        
 12  City                999 non-null    object        
 13  State               999 non-null    object        
 14

In [134]:
customer = ykdf[['Customer_ID', 'Customer_Name','Email', 'Phone_Number']].copy().drop_duplicates().reset_index(drop=True)


In [139]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 989 entries, 0 to 988
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Customer_ID    989 non-null    object
 1   Customer_Name  989 non-null    object
 2   Email          989 non-null    object
 3   Phone_Number   989 non-null    object
dtypes: object(4)
memory usage: 31.0+ KB


In [135]:
product = ykdf[['Product_ID','Product_Name', 'Brand', 'Category', 'Price',]].copy().drop_duplicates().reset_index(drop=True)

In [136]:
order = ykdf[['Order_ID', 'Customer_ID', 'Product_ID', 'Order_Date', 'Quantity', 'Total_Price']].copy().drop_duplicates().reset_index(drop=True)

In [137]:
shipping_address = ykdf[['Customer_ID','Shipping_Address','City', 'State','Country','Postal_Code']].copy().drop_duplicates().reset_index(drop=True)

In [168]:
shipping_address.index.name = 'Shipping_ID'
shipping_address.reset_index(inplace=True)
shipping_address.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 989 entries, 0 to 988
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Shipping_ID       989 non-null    int64 
 1   Customer_ID       989 non-null    object
 2   Shipping_Address  989 non-null    object
 3   City              989 non-null    object
 4   State             989 non-null    object
 5   Country           989 non-null    object
 6   Postal_Code       989 non-null    int64 
dtypes: int64(2), object(5)
memory usage: 54.2+ KB


In [138]:
payment_method = ykdf[['Order_ID', 'Payment_Method', 'Transaction_Status']].copy().drop_duplicates().reset_index(drop=True)

In [169]:
# exporting all to csv files
product.to_csv(os.path.join(processed, 'products.csv'), index=False)
customer.to_csv(os.path.join(processed, 'customers.csv'), index=False)   
order.to_csv(os.path.join(processed, 'orders.csv'), index=False)
shipping_address.to_csv(os.path.join(processed, 'shipping_address.csv'), index=False)
payment_method.to_csv(os.path.join(processed, 'payment_method.csv'), index=False)

In [141]:
def getdb_connection():
    conn = psycopg2.connect(
        dbname='yanki_ecommerce',
        user='postgres',
        password='password',
        host='localhost',
        port='5432'
    )
    return conn



In [191]:
def create_tables(sql_path):
    conn =getdb_connection()
    cursor = conn.cursor()


    # Read the SQL file
    with open(sql_path, 'r') as file:
        sql_commands = file.read()
    create_table_query = f" CREATE SCHEMA IF NOT EXISTS yanki; \
        {sql_commands} \
            "
    cursor.execute(create_table_query)
    conn.commit()
    cursor.close()
    conn.close()
    print(f"Tables created successfully {sql_path.split('/')[-1]}")


for sql_file in os.listdir(os.path.join('scripts')):
    # Check if the file ends with .sql
    if sql_file.endswith('.sql'):
        sql_file = os.path.join('scripts', sql_file)
        create_tables(sql_file)
        # print(f"Creating table from {sql_file}")
 

Tables created successfully products.sql
Tables created successfully customers.sql
Tables created successfully shipping_address.sql
Tables created successfully orders.sql
Tables created successfully payment_method.sql


In [192]:
def genPercentSym(data:tuple) -> str:

        d = []
        data = data.split(',')

        for i in range(len(data)):
            d.append('%s')


        return ', '.join(d)

In [193]:
import csv 

def load_data_from_csv(csv_path):
    conn = getdb_connection()
    cursor = conn.cursor()
    columns = None
    query = None
    tablename = os.path.basename(csv_path).split('/')[-1].split('.')[0]
    schema = 'yanki'
    
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
         # Skip header row
        # next(reader)
        for i,row in enumerate(reader):
            if i == 0:
                columns = ', '.join(row)
                continue
            else:
                
                query = f"INSERT INTO {schema}.{tablename} ({columns}) VALUES ({genPercentSym(columns)})"
                cursor.execute(query, row)
            

    conn.commit()
    cursor.close()
    conn.close()
    print(f"Data loaded from {csv_path} successfully.")

customers = os.path.join(processed, 'payment_method.csv')

# Load data into the database
for csv_file in os.listdir(os.path.join(processed)):
    if csv_file.endswith('.csv'):
        csv_path = os.path.join(processed, csv_file)
        load_data_from_csv(csv_path)



Data loaded from ./data/processed/customers.csv successfully.
Data loaded from ./data/processed/products.csv successfully.
Data loaded from ./data/processed/orders.csv successfully.
Data loaded from ./data/processed/shipping_address.csv successfully.
Data loaded from ./data/processed/payment_method.csv successfully.
